<a href="https://colab.research.google.com/github/K-lin99/COMP472-AI-Group-Project-/blob/mobilenetv2-hyperparameter-tuning/mobilenet_v2_brain_tumor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MobileNet V2

## 1. Setup

In [6]:
!pip install torchmetrics grad-cam --quiet
!pip install optuna --quiet

In [9]:
import copy, time, random, warnings, shutil, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchmetrics import Accuracy, Precision, Recall, F1Score
from sklearn.metrics import classification_report, confusion_matrix
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from google.colab import drive

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Fix for 'Mountpoint must not already contain files'
mount_path = '/content/drive'
if os.path.exists(mount_path) and not os.path.ismount(mount_path):
    print(f'Removing local non-mount directory: {mount_path}')
    shutil.rmtree(mount_path)

# Mount Google Drive
drive.mount(mount_path, force_remount=True)

OUTPUT_DIR = Path('/content/drive/MyDrive/COMP472/outputs/mobilenet_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Device: cpu
Removing local non-mount directory: /content/drive
Mounted at /content/drive


## 2. Hyperparameters

In [10]:
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
IMG_SIZE      = 224

## 3. Data (To fill)

Replace each `None` with the DataLoader.  
Images must be tensors of shape `(batch, 3, 224, 224)`, normalized with:
```
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```
(This can be changed if needed)

### 3.1 Load Datasets

In [11]:
from torch.utils.data import DataLoader

preprocessed_dir = '/content/drive/MyDrive/brain_mri_preprocessed'


# ── Dataset 1 : 3,264 images - 4 classes ──────────────────────────
train_data_ds1 = torch.load(f'{preprocessed_dir}/dataset1_train.pt')
val_data_ds1   = torch.load(f'{preprocessed_dir}/dataset1_val.pt')
test_data_ds1  = torch.load(f'{preprocessed_dir}/dataset1_test.pt')

train_images_ds1 = train_data_ds1['images']
train_labels_ds1 = train_data_ds1['labels']
val_images_ds1   = val_data_ds1['images']
val_labels_ds1   = val_data_ds1['labels']
test_images_ds1  = test_data_ds1['images']
test_labels_ds1  = test_data_ds1['labels']
class_names_ds1 = list(train_data_ds1['class_to_label'].keys())

# ── Dataset 2 : 7,023 images - 4 classes ──────────────────────────
train_data_ds2 = torch.load(f'{preprocessed_dir}/dataset2_train.pt')
val_data_ds2   = torch.load(f'{preprocessed_dir}/dataset2_val.pt')
test_data_ds2  = torch.load(f'{preprocessed_dir}/dataset2_test.pt')

train_images_ds2 = train_data_ds2['images']
train_labels_ds2 = train_data_ds2['labels']
val_images_ds2   = val_data_ds2['images']
val_labels_ds2   = val_data_ds2['labels']
test_images_ds2  = test_data_ds2['images']
test_labels_ds2  = test_data_ds2['labels']

class_names_ds2 = list(train_data_ds1['class_to_label'].keys())

# ── Dataset 3 : 4,478 images - 44 classes ─────────────────────────
train_data_ds3 = torch.load(f'{preprocessed_dir}/dataset3_train.pt')
val_data_ds3   = torch.load(f'{preprocessed_dir}/dataset3_val.pt')
test_data_ds3  = torch.load(f'{preprocessed_dir}/dataset3_test.pt')

train_images_ds3 = train_data_ds3['images']
train_labels_ds3 = train_data_ds3['labels']
val_images_ds3   = val_data_ds3['images']
val_labels_ds3   = val_data_ds3['labels']
test_images_ds3  = test_data_ds3['images']
test_labels_ds3  = test_data_ds3['labels']

class_names_ds3 = list(train_data_ds3['class_to_label'].keys())

### 3.2 Data Augmentation

In [12]:
# Monai imports

!python -c "import monai" || pip install -q "monai-weekly[ignite, tqdm]"

import logging
import numpy as np
import os
from pathlib import Path
import sys
import tempfile
import torch

from monai.apps import MedNISTDataset
from monai.config import print_config
from monai.engines import SupervisedTrainer
from monai.handlers import StatsHandler
from monai.inferers import SimpleInferer
from monai.networks import eval_mode
from monai.networks.nets import densenet121
from monai.transforms import (
    Compose,
    LoadImage,
    EnsureChannelFirst,
    Resize,
    NormalizeIntensity,
    RandFlipd,
    RandRotate,
    RandZoom,
    RandGaussianNoise,
    RandAdjustContrast,
    RandShiftIntensity,
    EnsureType,
)
from monai.data import ImageDataset


print_config()

# Define Augmentation class and transformation
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Convert 1-channel to 3-channel if necessary
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)

        if self.transform:
            image = self.transform(image)
        return image, label

def get_train_transforms():
    return Compose([
        RandRotate(range_x=0.15, prob=0.5, keep_size=True),
        RandZoom(min_zoom=0.95, max_zoom=1.05, prob=0.3, keep_size=True),
        RandGaussianNoise(mean=0.0, std=0.01, prob=0.2),
        RandAdjustContrast(gamma=(0.9, 1.1), prob=0.2),
        RandShiftIntensity(offsets=0.05, prob=0.2),
    ])



# ── Dataset 1 : AugmentDataset  ──────────────────────────
train_dataset_ds1 = AugmentedDataset(train_images_ds1, train_labels_ds1, transform=get_train_transforms())
val_dataset_ds1   = AugmentedDataset(val_images_ds1,   val_labels_ds1,   transform=None)
test_dataset_ds1  = AugmentedDataset(test_images_ds1,  test_labels_ds1,  transform=None)

# ── Dataset 2 : AugmentDataset  ──────────────────────────

train_dataset_ds2 = AugmentedDataset(train_images_ds2, train_labels_ds2, transform=get_train_transforms())
val_dataset_ds2   = AugmentedDataset(val_images_ds2,   val_labels_ds2,   transform=None)
test_dataset_ds2  = AugmentedDataset(test_images_ds2,  test_labels_ds2,  transform=None)

# ── Dataset 3 : AugmentDataset  ──────────────────────────

train_dataset_ds3 = AugmentedDataset(train_images_ds3, train_labels_ds3, transform=get_train_transforms())
val_dataset_ds3   = AugmentedDataset(val_images_ds3,   val_labels_ds3,   transform=None)
test_dataset_ds3  = AugmentedDataset(test_images_ds3,  test_labels_ds3,  transform=None)

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'monai'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 47.4 MB/s eta 0:00:00
MONAI version: 1.6.dev2612
Numpy version: 2.0.2
Pytorch version: 2.10.0+cpu
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 42d23a410d875c107bc893df5d77a5a2797dc2ef
MONAI __file__: /usr/local/lib/python3.12/dist-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: 0.4.11
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.4.2
scikit-image version: 0.25.2
scipy version: 1.16.3
Pillow version: 11.3.0
Tensorboard version: 2.19.0
gdown version: 5.2.1
TorchVision version: 0.25.0+cpu
tqdm version: 4.67.3
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 5.9.5
pandas version: 2.2.2
einops version: 0.8.2
transformers version: 5.0.0
ml

### 3.3 Data Loaders

In [13]:
# --- Assign to the expected loader variables ---

# ── Dataset 1 : assign Dataloaders ──────────────────────────
train_loader_ds1 = DataLoader(train_dataset_ds1, batch_size=BATCH_SIZE, shuffle=True)
val_loader_ds1   = DataLoader(val_dataset_ds1,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_ds1  = DataLoader(test_dataset_ds1,  batch_size=BATCH_SIZE, shuffle=False)
print(f"DS1 Train : {len(train_dataset_ds1)} images")
print(f"DS1 Val   : {len(val_dataset_ds1)} images")
print(f"DS1 Test  : {len(test_dataset_ds1)} images")
print(f"Classes   : {class_names_ds1}")

# ── Dataset 2 :  assign Dataloaders ──────────────────────────
train_loader_ds2 = DataLoader(train_dataset_ds2, batch_size=BATCH_SIZE, shuffle=True)
val_loader_ds2   = DataLoader(val_dataset_ds2,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_ds2  = DataLoader(test_dataset_ds2,  batch_size=BATCH_SIZE, shuffle=False)
print(f"DS2 Train : {len(train_dataset_ds2)} images")
print(f"DS2 Val   : {len(val_dataset_ds2)} images")
print(f"DS2 Test  : {len(test_dataset_ds2)} images")
print(f"Classes   : {class_names_ds2}")

# ── Dataset 3 :  assign Dataloaders ──────────────────────────

train_loader_ds3 = DataLoader(train_dataset_ds3, batch_size=BATCH_SIZE, shuffle=True)
val_loader_ds3   = DataLoader(val_dataset_ds3,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_ds3  = DataLoader(test_dataset_ds3,  batch_size=BATCH_SIZE, shuffle=False)
print(f"DS3 Train : {len(train_dataset_ds3)} images")
print(f"DS3 Val   : {len(val_dataset_ds3)} images")
print(f"DS3 Test  : {len(test_dataset_ds3)} images")
print(f"Classes   : {class_names_ds3}")


# ── DO NOT EDIT BELOW ─────────────────────────────────────────────
DATASETS = [
    {
        'name': 'Dataset 1 — Brain Tumor Classification (MRI)',
        'num_classes': 4, 'class_names': class_names_ds1,
        'train_loader': train_loader_ds1, 'val_loader': val_loader_ds1, 'test_loader': test_loader_ds1,
    },
    {
        'name': 'Dataset 2 — Brain Tumor MRI Dataset',
        'num_classes': 4, 'class_names': class_names_ds2,
        'train_loader': train_loader_ds2, 'val_loader': val_loader_ds2, 'test_loader': test_loader_ds2,
    },
    {
        'name': 'Dataset 3 — Brain Tumor MRI Images 44 Classes',
        'num_classes': 44, 'class_names': class_names_ds3,
        'train_loader': train_loader_ds3, 'val_loader': val_loader_ds3, 'test_loader': test_loader_ds3,
    },
]
print('Dataset config ready')

DS1 Train : 2236 images
DS1 Val   : 325 images
DS1 Test  : 655 images
Classes   : ['glioma', 'meningioma', 'pituitary', 'no_tumor']
DS2 Train : 5036 images
DS2 Val   : 720 images
DS2 Test  : 1444 images
Classes   : ['glioma', 'meningioma', 'pituitary', 'no_tumor']
DS3 Train : 3112 images
DS3 Val   : 432 images
DS3 Test  : 934 images
Classes   : ['Astrocitoma T1', 'Astrocitoma T1C+', 'Astrocitoma T2', 'Carcinoma T1', 'Carcinoma T1C+', 'Carcinoma T2', 'Ependimoma T1', 'Ependimoma T1C+', 'Ependimoma T2', 'Ganglioglioma T1', 'Ganglioglioma T1C+', 'Ganglioglioma T2', 'Germinoma T1', 'Germinoma T1C+', 'Germinoma T2', 'Glioblastoma T1', 'Glioblastoma T1C+', 'Glioblastoma T2', 'Granuloma T1', 'Granuloma T1C+', 'Granuloma T2', 'Meduloblastoma T1', 'Meduloblastoma T1C+', 'Meduloblastoma T2', 'Meningioma T1', 'Meningioma T1C+', 'Meningioma T2', 'Neurocitoma T1', 'Neurocitoma T1C+', 'Neurocitoma T2', 'Oligodendroglioma T1', 'Oligodendroglioma T1C+', 'Oligodendroglioma T2', 'Papiloma T1', 'Papiloma

## 4. Model - MobileNet V2

In [14]:
def build_mobilenet_v2(num_classes: int) -> nn.Module:
    """
    Final classifier head is replaced to match num_classes (4 or 44).
    """
    model = models.mobilenet_v2(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(DEVICE)

# Sanity check
_m = build_mobilenet_v2(4)
_x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
print(f'Output shape    : {list(_m(_x).shape)}')
print(f'Trainable params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}')
del _m, _x

Output shape    : [2, 4]
Trainable params: 2,228,996


## 5. Training & Evaluation Functions

In [15]:
def train_one_epoch(model, loader, criterion, optimizer, num_classes):
    model.train()
    loss_sum = 0.0
    acc_m = Accuracy(task='multiclass', num_classes=num_classes).to(DEVICE)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        acc_m.update(out.argmax(1), labels)
    return loss_sum / len(loader.dataset), acc_m.compute().item()


@torch.no_grad()
def evaluate(model, loader, criterion, num_classes):
    model.eval()
    loss_sum = 0.0
    acc_m = Accuracy(task='multiclass', num_classes=num_classes).to(DEVICE)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        loss_sum += criterion(out, labels).item() * imgs.size(0)
        acc_m.update(out.argmax(1), labels)
    return loss_sum / len(loader.dataset), acc_m.compute().item()


@torch.no_grad()
def full_evaluation(model, loader, num_classes, class_names):
    model.eval()
    all_preds, all_labels = [], []
    ms = {
        'acc':  Accuracy( task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'prec': Precision(task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'rec':  Recall(   task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'f1':   F1Score(  task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
    }
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        for m in ms.values(): m.update(preds, labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    results = {k: v.compute().item() for k, v in ms.items()}
    results.update({'all_preds': all_preds, 'all_labels': all_labels})

    print(f"  Accuracy : {results['acc']:.4f}")
    print(f"  Precision: {results['prec']:.4f}")
    print(f"  Recall   : {results['rec']:.4f}")
    print(f"  F1-Score : {results['f1']:.4f}")
    print(classification_report(all_labels, all_preds,
          target_names=(class_names if len(class_names) <= 20 else None), digits=4))
    return results


def train_model(model, cfg, save_path):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val, best_w = float('inf'), None
    nc = cfg['num_classes']

    for ep in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, cfg['train_loader'], criterion, optimizer, nc)
        vl, va = evaluate(model, cfg['val_loader'], criterion, nc)
        scheduler.step(vl)
        for k, v in zip(history, [tl, vl, ta, va]): history[k].append(v)
        flag = ''
        if vl < best_val:
            best_val, best_w = vl, copy.deepcopy(model.state_dict())
            torch.save(best_w, save_path)
            flag = ' ✅'
        print(f'Ep [{ep:>2}/{NUM_EPOCHS}] Train {tl:.4f}/{ta:.4f} | Val {vl:.4f}/{va:.4f} | {time.time()-t0:.1f}s{flag}')

    model.load_state_dict(best_w)
    return model, history


print('Training & evaluation functions ready')

Training & evaluation functions ready


## 6. Visualization Functions

In [16]:
def plot_curves(history, name, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'MobileNet V2 — {name}', fontweight='bold')
    for ax, key, title in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
        ax.plot(epochs, history[f'train_{key}'], 'b-o', ms=4, label='Train')
        ax.plot(epochs, history[f'val_{key}'],   'r-o', ms=4, label='Val')
        ax.set_title(title); ax.set_xlabel('Epoch')
        ax.legend(); ax.grid(alpha=0.3)
        if key == 'acc': ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_confusion_matrix(results, class_names, name, save_path):
    cm = confusion_matrix(results['all_labels'], results['all_preds'], normalize='true')
    n  = len(class_names)
    fig, ax = plt.subplots(figsize=(min(n*0.7+2, 20), min(n*0.6+2, 18)))
    sns.heatmap(cm, annot=(n<=15), fmt=('.2f' if n<=15 else ''), cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'Confusion Matrix — {name}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right', fontsize=(8 if n>15 else 10))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_grad_cam(model, loader, class_names, name, save_path, n=8):
    model.eval()
    cam    = GradCAM(model=model, target_layers=[model.features[-1]])
    imgs, labels = next(iter(loader))
    imgs   = imgs[:n].to(DEVICE)
    with torch.no_grad():
        preds = model(imgs).argmax(1).cpu()
    mean, std = np.array([0.485,0.456,0.406]), np.array([0.229,0.224,0.225])
    fig, axes = plt.subplots(2, n//2, figsize=(n*2, 8))
    fig.suptitle(f'Grad-CAM — {name}', fontweight='bold')
    for i, ax in enumerate(axes.flatten()):
        gc = cam(input_tensor=imgs[i:i+1], targets=[ClassifierOutputTarget(preds[i].item())])[0]
        img_np = np.clip(std * imgs[i].cpu().numpy().transpose(1,2,0) + mean, 0, 1).astype(np.float32)
        ax.imshow(show_cam_on_image(img_np, gc, use_rgb=True))
        ax.set_title(f'True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}',
                     color=('green' if preds[i]==labels[i] else 'red'), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


print('Visualization functions ready')

Visualization functions ready


## 7. Hyperparameter Tuning with Optuna

Before training across all datasets, we use Optuna to find the best `LEARNING_RATE` and `LOSS_FUNCTION` using Dataset 1 as a benchmark.

In [17]:
import optuna

def objective(trial):
  lr = trial.suggest_float("learning_rate", 1e-4, 9e-4, log=True)
  loss_type = trial.suggest_categorical("loss_fn", ["CrossEntropy", "LabelSmoothing"])

  cfg = DATASETS[0]
  if cfg["train_loader"] is None:
    return 0.0

  model = build_mobilenet_v2(cfg["num_classes"])

  if loss_type == "CrossEntropy":
    criterion = nn.CrossEntropyLoss()
  else:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

  optimizer = optim.Adam(model.parameters(), lr=lr)

  tuning_epochs = 5
  best_val_acc = 0.0

  for epoch in range(tuning_epochs):
    train_one_epoch(model, cfg["train_loader"], criterion, optimizer, cfg["num_classes"])
    _, val_acc = evaluate(model, cfg["val_loader"], criterion, cfg["num_classes"])

    if val_acc > best_val_acc:
      best_val_acc = val_acc

    trial.report(val_acc, epoch)
    if trial.should_prune():
      raise optuna.exceptions.TrialPruned()

  return best_val_acc

study = optuna.create_study(direction="maximize")

try:
  study.optimize(objective, n_trials=10)
  print("\nOptimization complete")
  print("Best params: ", study.best_params)
  LEARNING_RATE = study.best_params['learning_rate']
  print(f"Global LEARNING_RATE updated to: {LEARNING_RATE}")
except Exception as e:
  print(f"Optuna trial failed: {e}. Ensure DataLoaders are initialized in Section 3.")

[I 2026-03-25 23:43:08,625] A new study created in memory with name: no-name-4f1e3b9e-5032-4d17-94d3-ead67e6fa9cd
[W 2026-03-25 23:57:51,722] Trial 0 failed with parameters: {'learning_rate': 0.00012678346592512202, 'loss_fn': 'LabelSmoothing'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_10004/3947922706.py", line 24, in objective
    train_one_epoch(model, cfg["train_loader"], criterion, optimizer, cfg["num_classes"])
  File "/tmp/ipykernel_10004/2330624849.py", line 10, in train_one_epoch
    loss.backward()
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 621, in backward
    return handle_torch_function(
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/overrides.py", line 1763, in handle_to

KeyboardInterrupt: 

## 7. Train on All 3 Datasets

In [ ]:
all_results = {}

for i, cfg in enumerate(DATASETS, start=1):
    print(f'\n{"="*55}\n  Dataset {i}/3: {cfg["name"]}\n  Classes: {cfg["num_classes"]}\n{"="*55}')

    model    = build_mobilenet_v2(cfg['num_classes'])
    ckpt     = OUTPUT_DIR / f'mobilenet_v2_dataset{i}_best.pth'
    model, history = train_model(model, cfg, ckpt)

    print('\n── Test Results ──')
    results  = full_evaluation(model, cfg['test_loader'], cfg['num_classes'], cfg['class_names'])
    all_results[cfg['name']] = results

    print('\n── Plots ──')
    plot_curves(history, cfg['name'],           OUTPUT_DIR / f'mobilenet_v2_dataset{i}_curves.png')
    plot_confusion_matrix(results, cfg['class_names'], cfg['name'], OUTPUT_DIR / f'mobilenet_v2_dataset{i}_confusion.png')
    plot_grad_cam(model, cfg['test_loader'],    cfg['class_names'], cfg['name'], OUTPUT_DIR / f'mobilenet_v2_dataset{i}_gradcam.png')

    del model; torch.cuda.empty_cache()

print('\n All datasets complete!')

In [ ]:
!pip install fpdf --quiet

In [ ]:
from fpdf import FPDF
import datetime
from pathlib import Path
import os

# Ensure OUTPUT_DIR is defined
OUTPUT_DIR = Path('/content/drive/MyDrive/COMP472/outputs/mobilenet_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class PDFReport(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 15)
        self.cell(0, 10, 'MobileNet V2 - Brain Tumor Classification Report', 0, 1, 'C')
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')

def create_report(all_results, output_path, output_dir):
    pdf = PDFReport()
    pdf.set_auto_page_break(auto=True, margin=15)

    # Title Page
    pdf.add_page()
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 10, f'Date: {datetime.date.today()}', 0, 1)
    pdf.ln(5)

    # 1. Hyperparameters Section
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, 'Global Hyperparameters', 0, 1)
    pdf.ln(2)
    pdf.set_font('Arial', '', 12)

    hparams = {
        'Batch Size': BATCH_SIZE if 'BATCH_SIZE' in globals() else 'N/A',
        'Epochs': NUM_EPOCHS if 'NUM_EPOCHS' in globals() else 'N/A',
        'Learning Rate': f"{LEARNING_RATE:.6f}" if 'LEARNING_RATE' in globals() else 'N/A',
        'Weight Decay': WEIGHT_DECAY if 'WEIGHT_DECAY' in globals() else 'N/A',
        'Image Size': f"{IMG_SIZE}x{IMG_SIZE}" if 'IMG_SIZE' in globals() else 'N/A'
    }

    for k, v in hparams.items():
        pdf.cell(0, 8, f"- {k}: {v}", 0, 1)

    pdf.ln(10)

    # 2. Dataset Specific Results
    for i, (name, r) in enumerate(all_results.items(), start=1):
        pdf.set_font('Arial', 'B', 16)
        pdf.cell(0, 10, f'Dataset {i}: {name}', 0, 1)
        pdf.ln(5)

        # Metrics Table
        pdf.set_font('Arial', 'B', 12)
        pdf.cell(40, 10, 'Metric', 1)
        pdf.cell(40, 10, 'Value', 1, 1)

        pdf.set_font('Arial', '', 12)
        pdf.cell(40, 10, 'Accuracy', 1)
        pdf.cell(40, 10, f"{r['acc']:.4f}", 1, 1)
        pdf.cell(40, 10, 'Precision', 1)
        pdf.cell(40, 10, f"{r['prec']:.4f}", 1, 1)
        pdf.cell(40, 10, 'Recall', 1)
        pdf.cell(40, 10, f"{r['rec']:.4f}", 1, 1)
        pdf.cell(40, 10, 'F1-Score', 1)
        pdf.cell(40, 10, f"{r['f1']:.4f}", 1, 1)
        pdf.ln(10)

        # Embed Visualizations
        curve_path = str(output_dir / f'mobilenet_v2_dataset{i}_curves.png')
        if os.path.exists(curve_path):
            pdf.set_font('Arial', 'B', 12)
            pdf.cell(0, 10, 'Training & Validation Curves', 0, 1)
            pdf.image(curve_path, x=10, w=180)
            pdf.ln(5)

        cm_path = str(output_dir / f'mobilenet_v2_dataset{i}_confusion.png')
        if os.path.exists(cm_path):
            pdf.cell(0, 10, 'Confusion Matrix', 0, 1)
            pdf.image(cm_path, x=10, w=140)
            pdf.add_page()

        cam_path = str(output_dir / f'mobilenet_v2_dataset{i}_gradcam.png')
        if os.path.exists(cam_path):
            pdf.cell(0, 10, 'Grad-CAM Interpretability', 0, 1)
            pdf.image(cam_path, x=10, w=180)

        if i < len(all_results):
            pdf.add_page()

    pdf.output(output_path)
    print(f'Report with Hyperparameters saved to: {output_path}')

# Generate a unique filename
timestamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M')
report_path = str(OUTPUT_DIR / f'MobileNetV2_Report_{timestamp}.pdf')

try:
    create_report(all_results, report_path, OUTPUT_DIR)
except NameError as e:
    print(f"Error: {e}. Please ensure the training loop (Section 7) has finished to populate 'all_results'.")

## 8. Results Summary

In [ ]:
rows = [
    {'Dataset': name, 'Accuracy': f"{r['acc']:.4f}", 'Precision': f"{r['prec']:.4f}",
     'Recall': f"{r['rec']:.4f}", 'F1-Score': f"{r['f1']:.4f}"}
    for name, r in all_results.items()
]
df = pd.DataFrame(rows, index=[f'Dataset {i+1}' for i in range(len(rows))])
print(df.to_string())
df.to_csv(OUTPUT_DIR / 'mobilenet_v2_results.csv')
print(f'\nSaved to {OUTPUT_DIR}/mobilenet_v2_results.csv')